# Persian G2P (Grapheme-to-Phoneme) with ByT5-small on KaamelDict

This notebook fine-tunes `google/byt5-small` to convert Persian text (orthography) into space-separated phonetic representations (IPA/phonemes) using the **KaamelDict** dataset (~116k entries).

### Key Technical Settings:
- **Full Precision (FP32)**: Avoids known `fp16` underflow / NaN loss issues on T5/ByT5 architectures.
- **Gradient Accumulation (steps=2)**: Maintains effective batch size of 32 while preserving VRAM headroom.
- **Fast Evaluation**: `predict_with_generate=False` during epoch evaluation to prevent slow autoregressive beam searches.
- **Google Drive Persistence**: Checkpoints and the final model are saved directly to your 5TB Google Drive.

In [7]:
# Step 1: Install Required Libraries
!pip install -q transformers datasets accelerate evaluate pandas pyarrow

In [8]:
# Step 2: Check GPU and Mount Google Drive
import os
import torch
from google.colab import drive

assert torch.cuda.is_available(), "No GPU found! Go to Runtime > Change runtime type > Select T4 GPU."
print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# Mount Google Drive
drive.mount('/content/drive')

# Destination path inside your 5TB Google Drive
GDRIVE_SAVE_DIR = "/content/drive/MyDrive/models/persian-byt5-g2p"
os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)
print(f"Models will be saved permanently to: {GDRIVE_SAVE_DIR}")

Active GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models will be saved permanently to: /content/drive/MyDrive/models/persian-byt5-g2p


In [9]:
# Step 3: Load and Preprocess KaamelDict Dataset
import ast
import pandas as pd
from datasets import Dataset

print("Loading KaamelDict CSV dataset from Hugging Face...")
CSV_URL = "https://huggingface.co/datasets/MahtaFetrat/KaamelDict/resolve/main/KaamelDict.csv"
df = pd.read_csv(CSV_URL)

def format_phonemes(phoneme_raw):
    if pd.isna(phoneme_raw):
        return ""
    try:
        data = ast.literal_eval(str(phoneme_raw))
        if isinstance(data, list) and len(data) > 0:
            # Take primary pronunciation (first tuple) and space-separate
            return " ".join(data[0])
    except Exception:
        pass
    return ""

df["target"] = df["phoneme"].apply(format_phonemes)
df = df.dropna(subset=["grapheme"])
df = df[df["target"].str.strip() != ""][["grapheme", "target"]].rename(columns={"grapheme": "input"})
df["input"] = df["input"].astype(str)

print(f"Total valid phonetic pairs: {len(df):,}")
print("Sample entries:")
display(df.head(5))

# Train/Validation Split (95% Train, 5% Test)
raw_dataset = Dataset.from_pandas(df).train_test_split(test_size=0.05, seed=42)
print(f"Train size: {len(raw_dataset['train']):,} | Eval size: {len(raw_dataset['test']):,}")

Loading KaamelDict CSV dataset from Hugging Face...
Total valid phonetic pairs: 116,609
Sample entries:


,input,target
0,واترپولو,v A t e r p o l o
1,دولوکس,d o l u k s
2,شستشو‌‌دهنده,S o s t e S u d a h a n d e
3,جوشی,j u S i
4,فولاد,f u l A d


Train size: 110,778 | Eval size: 5,831


In [10]:
# Step 4: Tokenization via ByT5 (Byte-Level)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "google/byt5-small"
print(f"Loading tokenizer and model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    # Max 64 UTF-8 bytes for Persian input word
    model_inputs = tokenizer(examples["input"], max_length=64, truncation=True)
    # Max 128 UTF-8 bytes for space-separated phoneme output
    labels = tokenizer(examples["target"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_datasets = raw_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_dataset["train"].column_names
)
print("Tokenization complete!")

Loading tokenizer and model: google/byt5-small...


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Tokenizing dataset...


Map:   0%|          | 0/110778 [00:00<?, ? examples/s]

Map:   0%|          | 0/5831 [00:00<?, ? examples/s]

Tokenization complete!


In [11]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./local_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,     # Effective batch size = 32
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=False,                        # FP32: Prevents T5/ByT5 NaN loss / underflow on T4
    bf16=False,
    predict_with_generate=False,       # Fast loss-only eval during training
    logging_steps=100,
    warmup_steps=300,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,        # Fixed: using processing_class instead of tokenizer
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [12]:
# Step 6: Start Training & Save to Google Drive
print("Starting fine-tuning on T4 GPU...")
trainer.train()

print(f"\nTraining complete! Exporting model to Google Drive at: {GDRIVE_SAVE_DIR}")
trainer.save_model(GDRIVE_SAVE_DIR)
tokenizer.save_pretrained(GDRIVE_SAVE_DIR)
print("Model and tokenizer successfully saved in Google Drive!")

Starting fine-tuning on T4 GPU...


Epoch,Training Loss,Validation Loss
1,0.134159,0.057536
2,0.097760,0.042612
3,0.074424,0.037550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete! Exporting model to Google Drive at: /content/drive/MyDrive/models/persian-byt5-g2p


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer successfully saved in Google Drive!


In [13]:
# Step 7: Live Inference Verification
def predict_g2p(word: str) -> str:
    inputs = tokenizer(word, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=2,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_words = [
    "دانشگاه",
    "کامپیوتر",
    "خوش‌آمدید",
    "هوش مصنوعی",
    "آسمان",
    "تلفظ",
    "واترپولو",
    "دربازکن"
]

print("\n--- G2P Inference Verification ---")
for w in test_words:
    print(f"{w:15} -> {predict_g2p(w)}")


--- G2P Inference Verification ---
دانشگاه         -> d A n e S g A h
کامپیوتر        -> k A m p i y u t e r
خوش‌آمدید       -> x o S A m a d i d
هوش مصنوعی      -> h u S e m a s n u ? i
آسمان           -> ? A s
تلفظ            -> t a l a f f o z
واترپولو        -> v A t e r p o l o
دربازکن         -> d a r b A z k o n


In [3]:
# ==============================================================================
# Publish Persian ByT5 G2P to Hugging Face Hub & Link to KaamelDict Dataset
# ==============================================================================
!pip install -q huggingface_hub

import os
from google.colab import drive
from huggingface_hub import HfApi, login

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Authenticate with your Hugging Face Write Token
HF_TOKEN = "YOUR_HF_WRITE_TOKEN"
login(token=HF_TOKEN)

# 3. Target Repository Details
HF_USERNAME = "AminMadani"
REPO_NAME = "persian-byt5-g2p"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
DRIVE_DIR = "/content/drive/MyDrive/models/persian-byt5-g2p"

api = HfApi(token=HF_TOKEN)

# 4. Create the public repository on Hugging Face
print(f"Creating repository: {REPO_ID} ...")
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

# 5. Upload model weights and tokenizer from Google Drive
print(f"Uploading model shards and tokenizer from Google Drive to Hugging Face Hub...")
api.upload_folder(
    folder_path=DRIVE_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload fine-tuned ByT5-small weights for Persian G2P"
)
print("Model files uploaded successfully!")

# 6. Generate the Model Card (README.md) with Dataset & Base Model Linking
model_card_content = f"""---
language:
- fa
license: gpl-3.0
library_name: transformers
pipeline_tag: text2text-generation
tags:
- g2p
- grapheme-to-phoneme
- persian
- farsi
- byt5
- phonetics
- speech
- tts
datasets:
- MahtaFetrat/KaamelDict
base_model: google/byt5-small
metrics:
- loss
model-index:
- name: {REPO_NAME}
  results:
  - task:
      type: text2text-generation
      name: Grapheme-to-Phoneme
    dataset:
      name: KaamelDict
      type: MahtaFetrat/KaamelDict
    metrics:
    - type: loss
      value: 0.03755
      name: Validation Loss
---

# Persian ByT5 G2P (Grapheme-to-Phoneme)

A lightweight byte-level sequence-to-sequence model for converting Persian orthography into space-separated phonetic representations (IPA/phonemes).

Fine-tuned on the comprehensive [KaamelDict](https://huggingface.co/datasets/MahtaFetrat/KaamelDict) dictionary (~116,600 Persian phonetic entries) using `google/byt5-small`.

---

## Model Highlights
- **Byte-Level Architecture (ByT5):** Operates directly on raw UTF-8 bytes, completely eliminating out-of-vocabulary (OOV) errors and understanding Persian morphological structures (prefixes, suffixes, stems).
- **Implicit Diacritics & Orthography:** Incurs unwritten short vowels (*harakat* / *ezafe*), gemination (*tashdid*), and historical Persian spelling rules (e.g. `خوش` -> `/x o S/`).
- **Low Validation Loss:** Achieved a validation loss of **0.0375** across unseen evaluation splits.
- **Lightweight & Fast:** ~300M parameters, ideal for real-time TTS pipelines (such as Piper TTS) on both CPU and consumer GPUs.

---

## Quickstart & Usage

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_ID = "{REPO_ID}"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def text_to_phonemes(text: str) -> str:
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=2,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Sample Persian test words
samples = ["دانشگاه", "کامپیوتر", "خوش‌آمدید", "هوش مصنوعی", "تلفظ"]
for word in samples:
    print(f"{{word}} -> {{text_to_phonemes(word)}}")
```

---

## Training Details
- **Base Architecture:** `google/byt5-small` (~300M parameters)
- **Dataset:** [MahtaFetrat/KaamelDict](https://huggingface.co/datasets/MahtaFetrat/KaamelDict) (95% Train, 5% Test)
- **Precision:** FP32 (Full Precision for numerical stability)
- **Batch Size:** 16 per device with 2 gradient accumulation steps (effective batch size = 32)
- **Optimizer & LR:** AdamW with learning rate 5e-4
- **Epochs:** 3 epochs (~10,386 training steps)
- **Final Validation Loss:** `0.03755`

---

## Citation & References

If you use this model or the underlying KaamelDict dictionary, please cite:

```bibtex
@inproceedings{{qharabagh2025llm,
  title={{LLM-Powered Grapheme-to-Phoneme Conversion: Benchmark and Case Study}},
  author={{Qharabagh, Mahta Fetrat and Dehghanian, Zahra and Rabiee, Hamid R}},
  booktitle={{ICASSP 2025-2025 IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP)}},
  pages={{1--5}},
  year={{2025}},
  organization={{IEEE}}
}}

@article{{xue2022byt5,
  title={{ByT5: Towards a token-free future with pre-trained byte-to-byte models}},
  author={{Xue, Linting and Barua, Aditya and Constant, Noah and Al-Rfou, Rami and Narang, Sharan and Kale, Mihir and Roberts, Adam and Raffel, Colin}},
  journal={{Transactions of the Association for Computational Linguistics}},
  volume={{10}},
  pages={{291--306}},
  year={{2022}}
}}
```
"""

# Write and upload README.md
README_PATH = "/content/README.md"
with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(model_card_content)

print("Uploading Model Card (README.md) with metadata link to KaamelDict...")
api.upload_file(
    path_or_fileobj=README_PATH,
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Add comprehensive Model Card with KaamelDict dataset citation and tags"
)

print("\n" + "="*70)
print(f"CONGRATULATIONS! Your model is published and linked at:")
print(f"👉 https://huggingface.co/{REPO_ID}")
print("="*70)

Mounted at /content/drive
Creating repository: AminMadani/persian-byt5-g2p ...
Uploading model shards and tokenizer from Google Drive to Hugging Face Hub...
Model files uploaded successfully!
Uploading Model Card (README.md) with metadata link to KaamelDict...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11689: UserWarning: Warnings while validating metadata in README.md:
- The pipeline tag "text2text-generation" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image


CONGRATULATIONS! Your model is published and linked at:
👉 https://huggingface.co/AminMadani/persian-byt5-g2p
